# Toulmin JSON → Excel Export

Reads `outputs/3_toulmin_consolidated.json` and `api_outputs/toulmin_abstracts.json`,
then produces an Excel file with columns:
`Element_ID | Component | LLM_Content | Cited_PMIDs | Abstract`

In [5]:
import json
import pandas as pd
from pathlib import Path

In [6]:
# ── Paths ──────────────────────────────────────────────────────────────────
TOULMIN_PATH  = Path("outputs/4_toulmin_consolidated.json")
ABSTRACTS_PATH = Path("api_outputs/toulmin_abstracts.json")
OUTPUT_PATH   = Path("outputs/4_toulmin_table.xlsx")

# Load files
with open(TOULMIN_PATH)  as f: toulmin  = json.load(f)
with open(ABSTRACTS_PATH) as f: abstracts_raw = json.load(f)

# Build PMID → abstract dict
abstract_lookup = {}
for entry in abstracts_raw:
    pmid  = str(entry["pmid"])
    title = entry.get("title", "")
    body  = entry.get("abstract", "")
    abstract_lookup[pmid] = f"PMID: {pmid}\n{title}\n\n{body}"

print(f"Loaded {len(abstract_lookup)} abstracts")
print(f"Loaded Toulmin argument for: '{toulmin['metadata']['target_claim']}'")

Loaded 21 abstracts
Loaded Toulmin argument for: 'Semaglutide induces significant weight loss in adults with obesity'


In [8]:
arg = toulmin["toulmin_argument"]

def format_abstract_cell(pmids: list) -> str:
    """Return concatenated abstract text for all PMIDs in a cell."""
    parts = []
    for pmid in pmids:
        text = abstract_lookup.get(str(pmid), f"PMID: {pmid}\n[Abstract not available]")
        parts.append(text)
    return "\n\n" + "─" * 60 + "\n\n".join(parts) if len(parts) > 1 else (parts[0] if parts else "")

rows = []

# ── DATA ───────────────────────────────────────────────────────────────────
for i, item in enumerate(arg.get("data", []), start=1):
    pmids = item.get("pmids", [])
    rows.append({
        "Element_ID":  f"D{i:02d}",
        "Component":   "DATA",
        "LLM_Content": item["finding"],
        "Cited_PMIDs": ", ".join(pmids),
        "Abstract":    format_abstract_cell(pmids),
    })

# ── WARRANT ────────────────────────────────────────────────────────────────
warrant = arg.get("warrant", {})
if warrant:
    pmids = warrant.get("pmids", [])
    rows.append({
        "Element_ID":  "W01",
        "Component":   "WARRANT",
        "LLM_Content": warrant.get("content", ""),
        "Cited_PMIDs": ", ".join(pmids),
        "Abstract":    format_abstract_cell(pmids),
    })

# ── BACKING ────────────────────────────────────────────────────────────────
for i, item in enumerate(arg.get("backing", []), start=1):
    pmids = item.get("pmids", [])
    rows.append({
        "Element_ID":  f"B{i:02d}",
        "Component":   "BACKING",
        "LLM_Content": item["evidence"],
        "Cited_PMIDs": ", ".join(pmids),
        "Abstract":    format_abstract_cell(pmids),
    })

# ── QUALIFIERS ─────────────────────────────────────────────────────────────
for i, item in enumerate(arg.get("qualifiers", []), start=1):
    pmids = item.get("pmids", [])
    rows.append({
        "Element_ID":  f"Q{i:02d}",
        "Component":   "QUALIFIER",
        "LLM_Content": item["qualifier"],
        "Cited_PMIDs": ", ".join(pmids),
        "Abstract":    format_abstract_cell(pmids),
    })

# ── REBUTTALS ──────────────────────────────────────────────────────────────
for i, item in enumerate(arg.get("rebuttals", []), start=1):
    pmids = item.get("pmids", [])
    rows.append({
        "Element_ID":  f"R{i:02d}",
        "Component":   "REBUTTAL",
        "LLM_Content": item["rebuttal"],
        "Cited_PMIDs": ", ".join(pmids),
        "Abstract":    format_abstract_cell(pmids),
    })

df = pd.DataFrame(rows, columns=["Element_ID", "Component", "LLM_Content", "Cited_PMIDs", "Abstract"])
print(f"Total rows: {len(df)}")
df[["Element_ID", "Component", "Cited_PMIDs"]].head(30)

Total rows: 32


,Element_ID,Component,Cited_PMIDs
0,D01,DATA,33567185
1,D02,DATA,36216945
2,D03,DATA,33667417
3,D04,DATA,33755728
4,D05,DATA,36578889
5,D06,DATA,38679221
6,D07,DATA,38016699
7,D08,DATA,38446869
8,W01,WARRANT,"33567185, 36578889, 38016699"
9,B01,BACKING,"33567185, 33667417, 33755728"


In [9]:
# ── Write to Excel ─────────────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Toulmin")

    # Basic formatting
    ws = writer.sheets["Toulmin"]
    col_widths = {
        "A": 10,   # Element_ID
        "B": 12,   # Component
        "C": 80,   # LLM_Content
        "D": 30,   # Cited_PMIDs
        "E": 100,  # Abstract
    }
    for col, width in col_widths.items():
        ws.column_dimensions[col].width = width

    # Wrap text for content columns
    from openpyxl.styles import Alignment
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")

    # Bold header
    from openpyxl.styles import Font
    for cell in ws[1]:
        cell.font = Font(bold=True)

print(f"Saved → {OUTPUT_PATH.resolve()}")

Saved → /Users/ftzavellos/Law_and_Tech/drug_explanations/Code/outputs/4_toulmin_table.xlsx
